# Giải thích thuật toán Gợi ý Sản phẩm
## Dự án PJ-Selling Website

Notebook này trình bày chi tiết cách hoạt động của hai giải pháp gợi ý sản phẩm:
- **Solution 1**: Gợi ý sản phẩm liên quan dựa trên Co-buy & Category tương tự (áp dụng cho **mọi** sản phẩm).
- **Solution 2**: Gợi ý upsale cho riêng danh mục **Tã** (bỉm), hoạt động **trên nền** Solution 1.

> **Lưu ý**: Nội dung trong notebook này phản ánh đúng code đang chạy trong dự án, không phải logic cũ từ notebook gốc.

---
## 1. Các artifact đã tiền xử lý (Preprocessing)

Backend sử dụng dữ liệu đã được tiền xử lý bởi script `scripts/preprocess.py`, lưu trong thư mục `data/`:

| File | Mô tả |
|------|--------|
| `products.parquet` | Catalog sản phẩm đã làm sạch: giá, danh mục (l1/l2/l3), brand, manufacturer, metadata size cho tã (is_diaper, normalized_size, size_rank) |
| `item_cooccurrence.parquet` | Ma trận co-occurrence: mỗi dòng gồm (item_a, item_b, co_count) — số lần hai sản phẩm được mua chung |

### Cách tạo co-occurrence:
1. Gom transaction theo `(customer_id, ngày)` → mỗi nhóm là 1 **pseudo-session** (phiên mua hàng).
2. Giữ session có 2–50 item duy nhất (loại session quá lớn để tránh nổ tổ hợp).
3. Tạo tất cả cặp item `(A, B)` trong mỗi session, lưu cả 2 chiều `A→B` và `B→A`.
4. Đếm tần suất xuất hiện → `co_count`.

### Metadata size cho Tã:
- `normalized_size`: chuẩn hóa size về NB/S/M/L/XL/XXL/XXXL.
- `size_rank`: thứ hạng size (NB=0, S=1, M=2, L=3, XL=4, XXL=5, XXXL=6).
- Ưu tiên parse từ cột `size` gốc, nếu không có thì parse từ `description`.

---
## 2. Khái niệm Co-buy (Mua chung)

**Co-buy** (co-purchase) là khi hai sản phẩm thường xuyên được mua **cùng nhau** bởi cùng khách hàng trong cùng phiên mua hàng.

Ví dụ: Nếu 100 khách hàng mua sản phẩm A và B trong cùng đơn hàng, thì `co_count(A, B) = 100`.

**Ý nghĩa kinh doanh**: Sản phẩm có co_count cao cho thấy nhu cầu mua kèm mạnh — là cơ sở tốt để gợi ý.

---
## 3. Solution 1 — Co-buy + Lọc Category tương tự

### Luồng xử lý:

```
item_id → Lấy co-buy từ bảng co-occurrence
        → Join metadata category (l2, l3)
        → Lọc category tương tự (ưu tiên l3 > l2 > giữ nguyên)
        → Sắp xếp theo co_count giảm dần
        → Lấy top-N
```

### Chi tiết từng bước:

**Bước 1 — Lấy ứng viên co-buy:**
- Truy xuất tất cả dòng trong bảng co-occurrence có `item_a == item_id`.
- Mỗi dòng cho biết `item_b` (sản phẩm ứng viên) và `co_count` (số lần mua chung).

**Bước 2 — Join metadata danh mục:**
- Join với bảng products để lấy `category_l2` và `category_l3` của từng ứng viên.

**Bước 3 — Lọc theo danh mục tương tự (hierarchy):**
- **Ưu tiên 1**: Giữ ứng viên cùng `category_l3` với sản phẩm gốc.
- **Ưu tiên 2**: Nếu không có → giữ ứng viên cùng `category_l2`.
- **Fallback**: Nếu vẫn không → giữ nguyên toàn bộ tập co-buy (`co_buy_only`).

**Bước 4 — Xếp hạng:**
- Sắp xếp theo `co_count` giảm dần.
- Lấy top-N sản phẩm (mặc định N=20).

In [317]:
# Minh họa code Solution 1 (trích từ backend/app/services/Solution1.py)

def _filter_by_similar_category(candidates, product):
    """
    Lọc ứng viên co-buy theo danh mục sản phẩm gốc.
    Ưu tiên: category_l3 > category_l2 > giữ nguyên.
    """
    # Ưu tiên 1: cùng category_l3
    same_l3 = candidates.filter(pl.col("category_l3") == product["category_l3"])
    if same_l3.height > 0:
        return same_l3, "category_l3"

    # Ưu tiên 2: cùng category_l2
    same_l2 = candidates.filter(pl.col("category_l2") == product["category_l2"])
    if same_l2.height > 0:
        return same_l2, "category_l2"

    # Fallback: giữ nguyên tập co-buy
    return candidates, "co_buy_only"


def get_solution1_candidates(item_id):
    store = get_data_store()
    product = store.get_product(item_id)
    if product is None:
        return None, None, "not_found"

    # Bước 1: Lấy co-buy
    co_buy = store.cooccurrence.filter(pl.col("item_a") == item_id)

    # Bước 2: Join metadata
    candidates = co_buy.join(store.products.select("item_id", "category_l2", "category_l3"),
                             on="item_id", how="inner")

    # Bước 3: Lọc category
    filtered, strategy = _filter_by_similar_category(candidates, product)

    # Bước 4: Sắp xếp
    return product, filtered.sort("co_count", descending=True), f"solution1:co_buy+{strategy}"

---
## 4. Solution 2 — Upsale Tã theo Size

### Điều kiện áp dụng:
- **CHỈ** áp dụng cho sản phẩm có `category_l1 == "Tã"`.
- Sản phẩm không phải Tã → hệ thống **KHÔNG** gọi Solution 2.

### Luồng xử lý:

```
item_id (Tã) → Lấy tập ứng viên từ Solution 1
             → Lọc: chỉ giữ ứng viên thuộc danh mục Tã
             → Lọc: chỉ giữ ứng viên có size_rank >= size hiện tại
             → Tính size_gap = candidate_size_rank - current_size_rank
             → score_upsale = size_gap + 1
             → final_score = co_count × score_upsale
             → Sắp xếp theo final_score ↓, co_count ↓
             → Lấy top-N
```

### Chi tiết công thức chấm điểm:

| Biến | Công thức | Ý nghĩa |
|------|-----------|----------|
| `size_gap` | `candidate_size_rank - current_size_rank` | Chênh lệch bậc size (0 = cùng size, 1 = lớn hơn 1 bậc, ...) |
| `score_upsale` | `size_gap + 1` | Điểm ưu tiên upsale. Size lớn hơn → điểm cao hơn. |
| `final_score` | `co_count × score_upsale` | Kết hợp tần suất mua chung với mức ưu tiên upsale. |

### Bảng size_rank:

| Size | Rank |
|------|------|
| NB | 0 |
| S | 1 |
| M | 2 |
| L | 3 |
| XL | 4 |
| XXL | 5 |
| XXXL | 6 |

### Fallback:
- Nếu sản phẩm Tã không có `size_rank` → trả kết quả Solution 1.
- Nếu sau khi lọc không còn ứng viên tã hợp lệ → trả kết quả Solution 1.

In [316]:
# Minh họa code Solution 2 (trích từ backend/app/services/Solution2.py)

DIAPER_CATEGORY = "Tã"

def get_recommendations(item_id, n=20):
    store = get_data_store()

    # Bước 1: Lấy tập ứng viên từ Solution 1
    product, base_candidates, base_strategy = Solution1.get_solution1_candidates(item_id)

    # Bước 2: Không phải Tã → fallback Solution 1
    if product.get("category_l1") != DIAPER_CATEGORY:
        return _build_solution1_fallback_response(item_id, base_candidates, base_strategy, n)

    # Bước 3: Không có size_rank → fallback
    current_size_rank = product.get("size_rank")
    if current_size_rank is None:
        return _build_solution1_fallback_response(item_id, base_candidates, base_strategy, n,
                                                  reason="size_unknown")

    # Bước 4: Chấm điểm upsale
    scored = (
        base_candidates.select("item_id", "co_count")
        .join(store.products.select("item_id", "category_l1", "size_rank"), on="item_id", how="inner")
        .filter(pl.col("category_l1") == DIAPER_CATEGORY)
        .filter(pl.col("size_rank").is_not_null())
        .filter(pl.col("size_rank") >= int(current_size_rank))
        .with_columns((pl.col("size_rank") - int(current_size_rank)).alias("size_gap"))
        .with_columns((pl.col("size_gap") + 1).cast(pl.Float64).alias("score_upsale"))
        .with_columns((pl.col("co_count").cast(pl.Float64) * pl.col("score_upsale")).alias("final_score"))
    )

    # Bước 5: Xếp hạng
    result = scored.sort(by=["final_score", "co_count"], descending=[True, True]).head(n)

---
## 5. Tổng quan: Khi nào dùng Solution nào?

| Loại sản phẩm | Solution 1 | Solution 2 |
|----------------|------------|------------|
| **Sản phẩm thường** (không phải Tã) | ✅ Dùng duy nhất | ❌ Không áp dụng |
| **Tã (bỉm)** | ✅ Dùng làm nền (tập ứng viên) | ✅ Thêm layer upsale theo size |

### Trên giao diện web:
- **Sản phẩm thường**: Chỉ hiển thị carousel **"Related Products (Solution 1)"**.
- **Sản phẩm Tã**: Hiển thị **cả hai** carousel:
  - "Related Products (Solution 1)" — sản phẩm mua chung, lọc category.
  - "Recommended Products (Solution 2)" — upsale tã size lớn hơn.

---
## 6. Tại sao sản phẩm không phải Tã chỉ dùng Solution 1?

- Solution 2 **chỉ** thêm logic chấm điểm upsale dựa trên **size** (NB, S, M, L, XL, ...).
- Khái niệm size này chỉ có ý nghĩa với **Tã** (bỉm) — nơi trẻ em lớn dần và cần chuyển sang size lớn hơn.
- Các sản phẩm khác (sữa, đồ chơi, hóa mỹ phẩm, ...) không có khái niệm upsale theo size.
- Vì vậy, sản phẩm không phải Tã chỉ cần gợi ý dựa trên tần suất mua chung (co-buy) kết hợp lọc category tương tự → đó chính là Solution 1.

---
## 7. API Endpoints

| Route | Mô tả | Service |
|-------|--------|---------|
| `GET /api/related/{item_id}` | Sản phẩm liên quan (Solution 1) | `Solution1.get_related_products()` |
| `GET /api/recommendations/{item_id}` | Gợi ý upsale (Solution 2) | `Solution2.get_recommendations()` |
| `GET /api/products/{item_id}` | Thông tin sản phẩm | `product_service.get_product()` |

Frontend kiểm tra `product.category_l1 == "Tã"` để quyết định có gọi endpoint Solution 2 hay không.